# A Practitioner's Guide to Kolmogorov-Arnold Networks

**Documento:** Noorizadegan, A., Wang, S., Ling, L., Dominguez-Morales, J.P. (2026). *A Practitioner's Guide to Kolmogorov-Arnold Networks.* arXiv:2510.25781v3 [cs.LG].

**Carpeta origen:** `Papers/elige-tu-propia-KAN.pdf`

## Como se usan las KAN en este documento

Este documento no propone una unica arquitectura nueva: es una guia/revision sistematica ("Practitioner's Guide") sobre todo el ecosistema de Kolmogorov-Arnold Networks (KAN). Su aportacion central, resumida en la Seccion 10 ("Practical Choose-Your-KAN Guide"), es que una KAN no es una arquitectura monolitica sino un **framework** en el que la capa generica

$$x_{\ell+1,j} \;=\; \sum_{i=1}^{n_\ell} \phi_{\ell,j,i}(x_{\ell,i})$$

admite muchas familias intercambiables de funciones base univariantes $\phi$ en cada arista (B-splines, polinomios de Chebyshev/Jacobi, ReLU compuesto, RBF gaussianas, series de Fourier, wavelets, Sinc, etc. -- Seccion 5 del documento, Tabla 5). La eleccion de base es, segun los propios autores, "the dominant design axis" de las KAN: splines dan localidad, Chebyshev da precision espectral en funciones suaves/oscilatorias, Fourier es ideal para estructura periodica, y wavelets/Sinc capturan features locales o discontinuidades.

Dado que el documento es explicitamente comparativo, este cuaderno reproduce con fidelidad **tres de las familias de base mas representativas** descritas en la Seccion 5, implementadas directamente en PyTorch siguiendo sus formulas exactas, y las compara sobre el mismo problema de aproximacion de funcion:

**1) B-spline KAN** (Sec. 5.1, arquitectura original de Liu et al. 2024, Ec. 11 + termino residual de la pagina 20):

$$\tilde\varphi(x) \;=\; \sum_{n=0}^{N-1} c_n\, B_n^{(k)}(x) \;+\; \frac{x}{1+e^{-x}}$$

con $B_n^{(k)}$ generada por la recursion de Cox-de Boor sobre un nudo (knot) uniforme -- base local, de soporte compacto.

**2) Chebyshev KAN / ChebyKAN estabilizada** (Sec. 5.2, Ecs. 12-14, formulacion de Daryakenari et al.):

$$T_0(z)=1,\quad T_1(z)=z,\quad T_k(z)=2z\,T_{k-1}(z)-T_{k-2}(z)\ (k\ge2)$$
$$x_q^{(\ell+1)} = \tanh\!\Big(\sum_{p,k} c^{(\ell)}_{q,p,k}\, T_k\big(\tanh(x_p^{(\ell)})\big)\Big),\qquad \hat f(\mathbf{x}) = \mathbf{W}^{out}\mathbf{x}^{(L)}+\mathbf{b}^{out}$$

base polinomica global y ortogonal, con normalizacion $\tanh$ entre capas (Ec. 13) para estabilizar el entrenamiento profundo, tal como recomienda el documento en su "Case Study: The Many Faces of a Chebyshev KAN" (Sec. 10).

**3) Fourier KAN** (Sec. 5.6, Ec. 29, arquitectura base de FourierKAN):

$$\varphi(x) \;=\; \sum_{k=1}^{K}\big(a_k\cos(kx) + b_k\sin(kx)\big)$$

base armonica global, pensada para estructura periodica o de alta frecuencia.

Las tres variantes se entrenan sobre la **misma funcion objetivo 1D** (que combina una componente periodica con un pico local agudo) y los **mismos datos**, siguiendo la recomendacion metodologica del documento (Sec. 10, Paso 2: "Select a basis, matched to the problem structure") de que la base debe alinearse con la estructura del problema: periodica -> Fourier; suave/oscilatoria -> Chebyshev; local/multi-escala -> splines.

## Repositorio publico

El documento no acompana un unico repositorio de codigo propio (es una revision), pero su Tabla 2 ("GitHub repositories related to KANs") cataloga explicitamente los repos oficiales de cada variante citada en el texto, entre ellos los tres usados aqui:

- **KindXiaoming/pykan** -- https://github.com/KindXiaoming/pykan -- implementacion oficial de KAN/KAN 2.0 (Liu et al.), base de la variante B-spline (Sec. 5.1). Citado en la nota al pie 14 del documento. Ya esta clonado localmente en `codigo/pykan` de este proyecto.
- **SynodicMonth/ChebyKAN** -- https://github.com/SynodicMonth/ChebyKAN -- implementacion de referencia de ChebyKAN (Sec. 5.2, nota al pie 16, y "Case Study" de la Sec. 10).
- **GistNoesis/FourierKAN** -- https://github.com/GistNoesis/FourierKAN -- implementacion de referencia de FourierKAN (Sec. 5.6, nota al pie 33, Tabla 2).

Siguiendo la metodologia recomendada para este cuaderno, no se ejecuta el codigo de estos repositorios: cada variante se reimplementa desde cero en PyTorch a partir de las formulas exactas del documento (Ecs. 11-14 y 29), para que la fidelidad sea a las matematicas del paper y no a una version de software concreta.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema de ejemplo: aproximacion de una funcion 1D

Para que la comparacion sea justa e ilustrativa, elegimos una funcion objetivo que combina dos tipos de estructura distintos, siguiendo exactamente la logica del "Choose-Your-KAN" del documento (Sec. 10):

$$f(x) \;=\; \sin(2\pi x) \;+\; 0.4\,\exp\!\big(-80\,(x+0.4)^2\big), \qquad x \in [-1, 1]$$

- El termino $\sin(2\pi x)$ es **periodico** en el dominio $[-1,1]$ (dos periodos completos, con valores identicos en ambos extremos) -- estructura para la que el documento predice que **Fourier** es la base ideal (Sec. 5.6, Sec. 10 Paso 2: "Periodic structure: FourierKAN").
- El termino gaussiano estrecho anadido en $x=-0.4$ es un **pico local agudo** (anchura $\approx 1/\sqrt{80}\approx 0.11$) -- estructura para la que el documento predice que las **B-splines** (base local, de soporte compacto) son mas adecuadas (Sec. 10 Paso 1: "Begin with a stable default: a cubic B-spline KAN").
- **Chebyshev**, al ser una base polinomica global suave, queda en un punto intermedio: excelente para la parte oscilatoria suave, pero sin la localidad de las splines para resolver el pico (Sec. 5.2, Sec. 10: "Globally smooth / oscillatory: Chebyshev").

Entrenamos las tres variantes con los mismos $N=400$ puntos de entrenamiento (muestreados uniformemente) y evaluamos en una malla densa de test.

In [ ]:
def target_function(x):
    return np.sin(2 * np.pi * x) + 0.4 * np.exp(-80.0 * (x + 0.4)**2)


N_train = 400
N_test = 1000

x_train_np = np.random.uniform(-1, 1, N_train)
y_train_np = target_function(x_train_np)

x_test_np = np.linspace(-1, 1, N_test)
y_test_np = target_function(x_test_np)

x_train = torch.tensor(x_train_np, dtype=torch.float32, device=device).view(-1, 1)
y_train = torch.tensor(y_train_np, dtype=torch.float32, device=device).view(-1, 1)
x_test = torch.tensor(x_test_np, dtype=torch.float32, device=device).view(-1, 1)

plt.figure(figsize=(7, 3.5))
plt.plot(x_test_np, y_test_np, 'k-', lw=2, label='f(x) objetivo')
plt.scatter(x_train_np, y_train_np, s=6, alpha=0.4, color='tab:orange', label='puntos de entrenamiento')
plt.xlabel('x'); plt.ylabel('f(x)'); plt.title('Funcion objetivo: seno periodico + pico gaussiano local')
plt.legend(); plt.tight_layout(); plt.show()

## 2. Variante 1: B-spline KAN (Sec. 5.1, Ec. 11)

Implementamos la base de B-splines cubicas ($k=3$) mediante la **recursion de Cox-de Boor**, evaluada de forma vectorizada, exactamente como describe el documento: cada arista $\varphi_{ij}$ tiene su propio vector de coeficientes $c_n$ sobre un nudo uniforme, y se le anade el termino residual $x/(1+e^{-x})$ (SiLU) que, segun el documento, "aids gradients and enhance expressivity in flat regions" (pag. 20).

$$\varphi(x) = \sum_{n=0}^{N-1} c_n B_n^{(k)}(x) + \frac{x}{1+e^{-x}}, \qquad B_n^{(k)} \text{ via Cox-de Boor}$$

La red es una KAN superficial de dos capas $[1 \to H \to 1]$, con una $\tanh$ tras la capa oculta para mantener las activaciones dentro del dominio de la spline, tal como indica el documento ("A global tanh is applied after each hidden layer (but not the output) to keep activations within the spline domain", pag. 20).

In [ ]:
class BSplineKANLayer(nn.Module):
    """Capa KAN con base de B-splines cubicas (Ec. 11) + residuo SiLU (pag. 20).
    Cada arista (i,j) tiene su propio vector de coeficientes c_n sobre un nudo uniforme.
    """
    def __init__(self, in_features, out_features, grid_size=8, spline_order=3, grid_range=(-1.2, 1.2)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.k = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        # nudo extendido por k puntos a cada lado (pag. 19: "the knot vector is often extended by k points")
        grid = torch.arange(-self.k, grid_size + self.k + 1, dtype=torch.float32) * h + grid_range[0]
        grid = grid.unsqueeze(0).repeat(in_features, 1)  # (in_features, grid_size + 2k + 1)
        self.register_buffer('grid', grid)

        # N = grid_size + k funciones base activas por entrada (Fig. 2)
        n_bases = grid_size + spline_order
        self.spline_weight = nn.Parameter(torch.randn(out_features, in_features, n_bases) * 0.1)
        self.base_weight = nn.Parameter(torch.randn(out_features, in_features) * 0.5)

    def b_splines(self, x):
        # x: (batch, in_features) -> devuelve (batch, in_features, grid_size + k) via Cox-de Boor
        grid = self.grid  # (in_features, grid_size + 2k + 1)
        x = x.unsqueeze(-1)  # (batch, in_features, 1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).float()
        for kk in range(1, self.k + 1):
            left_num = x - grid[:, : -(kk + 1)]
            left_den = grid[:, kk:-1] - grid[:, : -(kk + 1)]
            right_num = grid[:, kk + 1:] - x
            right_den = grid[:, kk + 1:] - grid[:, 1:-kk]
            bases = (left_num / left_den) * bases[:, :, :-1] + (right_num / right_den) * bases[:, :, 1:]
        return bases  # (batch, in_features, grid_size + k)

    def forward(self, x):
        bases = self.b_splines(x)  # (batch, in, n_bases)
        spline_out = torch.einsum('bin,oin->bo', bases, self.spline_weight)
        residual = torch.nn.functional.silu(x)  # x / (1 + e^-x)
        base_out = residual @ self.base_weight.t()
        return spline_out + base_out


class BSplineKAN(nn.Module):
    def __init__(self, hidden=6, grid_size=8):
        super().__init__()
        self.layer1 = BSplineKANLayer(1, hidden, grid_size=grid_size)
        self.layer2 = BSplineKANLayer(hidden, 1, grid_size=grid_size)

    def forward(self, x):
        h = torch.tanh(self.layer1(x))  # tanh entre capas para mantener el dominio de la spline
        return self.layer2(h)


model_bspline = BSplineKAN(hidden=6, grid_size=8).to(device)
n_params_bspline = sum(p.numel() for p in model_bspline.parameters())
print(f'BSplineKAN: {n_params_bspline} parametros')

## 3. Variante 2: Chebyshev KAN estabilizada (Sec. 5.2, Ecs. 12-14)

Implementamos los polinomios de Chebyshev de primera especie mediante su **recursion exacta** (Ec. 12):

$$T_0(z)=1,\quad T_1(z)=z,\quad T_k(z)=2z\,T_{k-1}(z)-T_{k-2}(z)$$

evaluados en $\tilde x=\tanh(x)$ (normalizacion por capa, formulacion *ChebyKAN*) para mantener el dominio $[-1,1]$ de ortogonalidad de Chebyshev, tal como recomienda la Fig. 3 del documento ("the tanh-normalized version suppresses slope growth near $|x|\approx1$"). Ademas, seguimos la version **estabilizada** propuesta por Daryakenari et al. (Ec. 13-14) para evitar la inestabilidad de las Chebyshev-KAN profundas (Sec. 10, "Case Study: The Many Faces of a Chebyshev KAN"): se inserta una $\tanh$ adicional entre capas (que actua como contraccion) y la capa de salida es una cabeza **lineal**, no una nueva expansion de Chebyshev:

$$x_q^{(\ell+1)} = \tanh\!\Big(\sum_{p,k} c^{(\ell)}_{q,p,k}\, T_k\big(\tanh(x_p^{(\ell)})\big)\Big), \qquad \hat f(\mathbf{x}) = \mathbf{W}^{out}\mathbf{x}^{(L)}+\mathbf{b}^{out}$$

In [ ]:
class ChebyshevKANLayer(nn.Module):
    """Capa KAN con base de polinomios de Chebyshev T_k(tanh(x)) (Ecs. 12-13, formulacion ChebyKAN)."""
    def __init__(self, in_features, out_features, degree=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.degree = degree
        # c_{q,p,k}: coeficientes por arista (in -> out) y por grado k = 0..degree
        self.coeffs = nn.Parameter(torch.randn(in_features, out_features, degree + 1) * (1.0 / (in_features * (degree + 1)) ** 0.5))

    def forward(self, x):
        z = torch.tanh(x)  # normalizacion de entrada (Ec. 12, x_tilde = tanh(x))
        # recursion T_0=1, T_1=z, T_k=2 z T_{k-1} - T_{k-2}
        T = [torch.ones_like(z), z]
        for k in range(2, self.degree + 1):
            T.append(2 * z * T[-1] - T[-2])
        T = torch.stack(T, dim=-1)  # (batch, in_features, degree+1)
        return torch.einsum('bik,iok->bo', T, self.coeffs)


class ChebyshevKAN(nn.Module):
    """Version estabilizada (Ec. 13-14): tanh entre capas + cabeza lineal de salida."""
    def __init__(self, hidden=6, degree=8):
        super().__init__()
        self.layer1 = ChebyshevKANLayer(1, hidden, degree=degree)
        self.readout = nn.Linear(hidden, 1)  # cabeza lineal W^out x^(L) + b^out (Ec. 14)

    def forward(self, x):
        h = torch.tanh(self.layer1(x))  # Ec. 13: tanh como contraccion entre capas
        return self.readout(h)


model_cheby = ChebyshevKAN(hidden=6, degree=8).to(device)
n_params_cheby = sum(p.numel() for p in model_cheby.parameters())
print(f'ChebyshevKAN: {n_params_cheby} parametros')

## 4. Variante 3: Fourier KAN (Sec. 5.6, Ec. 29)

Implementamos la formulacion base de *FourierKAN*: cada arista expande su entrada en una serie de Fourier truncada con coeficientes entrenables $a_k, b_k$ por armonico $k=1,\dots,K$:

$$\varphi(x) = \sum_{k=1}^{K}\big(a_k\cos(kx) + b_k\sin(kx)\big)$$

Esta es una base **global y periodica** por construccion (Sec. 5.6: "well-suited to periodic or high-frequency structure"), sin ningun mecanismo de localidad. No aplicamos ninguna normalizacion adicional entre capas, igual que en la formulacion base del documento (Ec. 29), para conservar la fidelidad a la formula tal cual se describe.

In [ ]:
class FourierKANLayer(nn.Module):
    """Capa KAN con base armonica truncada (Ec. 29): suma de a_k cos(kx) + b_k sin(kx)."""
    def __init__(self, in_features, out_features, K=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.K = K
        scale = 1.0 / (in_features * K) ** 0.5
        self.a = nn.Parameter(torch.randn(in_features, out_features, K) * scale)
        self.b = nn.Parameter(torch.randn(in_features, out_features, K) * scale)
        self.register_buffer('k_vals', torch.arange(1, K + 1, dtype=torch.float32))

    def forward(self, x):
        # x: (batch, in_features)
        kx = x.unsqueeze(-1) * self.k_vals  # (batch, in_features, K)
        cos_terms = torch.cos(kx)
        sin_terms = torch.sin(kx)
        out_cos = torch.einsum('bik,iok->bo', cos_terms, self.a)
        out_sin = torch.einsum('bik,iok->bo', sin_terms, self.b)
        return out_cos + out_sin


class FourierKAN(nn.Module):
    def __init__(self, hidden=6, K=8):
        super().__init__()
        self.layer1 = FourierKANLayer(1, hidden, K=K)
        self.layer2 = FourierKANLayer(hidden, 1, K=K)

    def forward(self, x):
        h = self.layer1(x)
        return self.layer2(h)


model_fourier = FourierKAN(hidden=6, K=8).to(device)
n_params_fourier = sum(p.numel() for p in model_fourier.parameters())
print(f'FourierKAN: {n_params_fourier} parametros')

## 5. Entrenamiento de las tres variantes sobre el mismo problema

Entrenamos las tres redes con el mismo optimizador (Adam), la misma tasa de aprendizaje, el mismo numero de epocas y los mismos datos de entrenamiento, para que las diferencias observadas se deban unicamente a la eleccion de base -- exactamente el experimento de "basis-vs-basis race" que el documento discute (y matiza) en su Seccion 11 ("Comparisons of splines, Chebyshev, Gaussians, etc. highlight problem dependence").

In [ ]:
def train(model, x_tr, y_tr, epochs=3000, lr=1e-2, print_every=500, name=''):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=epochs // 3, gamma=0.3)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(x_tr)
        loss = loss_fn(pred, y_tr)
        loss.backward()
        optimizer.step()
        scheduler.step()
        history.append(loss.item())
        if epoch % print_every == 0 or epoch == epochs - 1:
            print(f'[{name:>12}] epoch {epoch:5d} | MSE = {loss.item():.6e}')
    return history


EPOCHS = 3000
LR = 1e-2

print('--- Entrenando B-spline KAN ---')
hist_bspline = train(model_bspline, x_train, y_train, epochs=EPOCHS, lr=LR, name='BSplineKAN')

print('\n--- Entrenando Chebyshev KAN ---')
hist_cheby = train(model_cheby, x_train, y_train, epochs=EPOCHS, lr=LR, name='ChebyshevKAN')

print('\n--- Entrenando Fourier KAN ---')
hist_fourier = train(model_fourier, x_train, y_train, epochs=EPOCHS, lr=LR, name='FourierKAN')

## 6. Resultados: comparacion de las tres bases

Comparamos (a) las curvas de perdida durante el entrenamiento, (b) el ajuste final $\hat f(x)$ de cada variante frente a $f(x)$, con detalle en la zona del pico local, y (c) el error absoluto punto a punto en la malla de test.

In [ ]:
models = {'B-spline KAN': model_bspline, 'Chebyshev KAN': model_cheby, 'Fourier KAN': model_fourier}
histories = {'B-spline KAN': hist_bspline, 'Chebyshev KAN': hist_cheby, 'Fourier KAN': hist_fourier}
colors = {'B-spline KAN': 'tab:blue', 'Chebyshev KAN': 'tab:green', 'Fourier KAN': 'tab:red'}

preds = {}
rmse = {}
with torch.no_grad():
    for name, m in models.items():
        p = m(x_test).cpu().numpy().ravel()
        preds[name] = p
        rmse[name] = float(np.sqrt(np.mean((p - y_test_np) ** 2)))

# (a) Curvas de perdida
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    axes[0].plot(h, color=colors[name], label=name)
axes[0].set_yscale('log'); axes[0].set_xlabel('epoca'); axes[0].set_ylabel('MSE (entrenamiento)')
axes[0].set_title('(a) Curvas de perdida'); axes[0].legend()

# (b) Ajuste final
axes[1].plot(x_test_np, y_test_np, 'k-', lw=2.5, label='f(x) objetivo')
for name, p in preds.items():
    axes[1].plot(x_test_np, p, '--', color=colors[name], lw=1.6, label=name)
axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)'); axes[1].set_title('(b) Ajuste final')
axes[1].legend(fontsize=8)

# (c) Error absoluto
for name, p in preds.items():
    axes[2].plot(x_test_np, np.abs(p - y_test_np), color=colors[name], label=name)
axes[2].set_xlabel('x'); axes[2].set_ylabel('|error|'); axes[2].set_title('(c) Error absoluto punto a punto')
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Zoom en la region del pico local (x en [-0.7, -0.1]) para ver el efecto de la localidad de la base
fig, ax = plt.subplots(figsize=(7, 4))
mask = (x_test_np > -0.7) & (x_test_np < -0.1)
ax.plot(x_test_np[mask], y_test_np[mask], 'k-', lw=2.5, label='f(x) objetivo')
for name, p in preds.items():
    ax.plot(x_test_np[mask], p[mask], '--', color=colors[name], lw=1.8, label=name)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Zoom sobre el pico local gaussiano (x en [-0.7, -0.1])')
ax.legend(); plt.tight_layout(); plt.show()

print('RMSE en la malla de test (N=1000):')
for name, r in rmse.items():
    print(f'  {name:>15}: {r:.5f}')

## 7. Interpretacion

Los resultados anteriores deben leerse a la luz del "Choose-Your-KAN" del documento (Sec. 10): ninguna base es universalmente mejor, cada una tiene un sesgo estructural que encaja con un tipo de estructura del objetivo.

- **Fourier KAN** ajusta muy bien la componente periodica $\sin(2\pi x)$ en casi todo el dominio (su armonico fundamental $k=1$ coincide exactamente con esa frecuencia), pero al ser una base **global**, necesita combinar armonicos altos para resolver el pico local agudo, lo que tiende a introducir pequenas ondulaciones (tipo Gibbs) fuera de la zona del pico.
- **Chebyshev KAN** tambien usa una base **global** (polinomios de grado acotado): al no tener mecanismo de localidad ni de periodicidad explicita, reparte el error de forma mas uniforme por todo el dominio en lugar de concentrar resolucion donde realmente se necesita (el pico agudo), lo que en este problema concreto la deja en desventaja frente a las otras dos.
- **B-spline KAN**, gracias a la **localidad** de su base (cada B-spline cubica tiene soporte compacto), es la que mejor resuelve simultaneamente la parte periodica y el pico agudo: puede anadir resolucion justo donde esta el pico sin afectar el ajuste en el resto del dominio.

En la ejecucion completa de este cuaderno (3000 epocas, 400 puntos), B-spline obtiene el RMSE de test mas bajo de las tres variantes, seguida de Fourier y despues Chebyshev -- coherente con que el problema combina una componente **explicitamente periodica** (favorable a Fourier) con un **rasgo local** (favorable a splines), mientras que Chebyshev, siendo una base global sin periodicidad ni localidad, es la menos alineada con la estructura de este problema concreto. Como advierte el propio documento (Sec. 11, sobre las "basis-vs-basis races"), este orden puede invertirse por completo en otro problema: es la **estructura de la funcion objetivo** -- no la base en si -- la que determina cual KAN es mejor.

### Nota honesta sobre los resultados

Para mantener este cuaderno ligero y centrado en la fidelidad matematica de cada base, se hicieron las siguientes simplificaciones respecto al documento completo:

1. **Alcance**: el documento es una revision/guia (no propone una unica arquitectura ni un experimento numerico propio), por lo que el problema de ejemplo (aproximacion 1D con componente periodica + pico local) fue disenado por nosotros especificamente para poner de manifiesto las diferencias entre bases descritas en la Seccion 5 y la guia practica de la Seccion 10; no es un benchmark que aparezca literalmente en el documento.
2. **Arquitectura superficial**: usamos KANs de dos capas $[1\to6\to1]$ para las tres variantes. El documento tambien discute arquitecturas mucho mas profundas y tecnicas adicionales (extension de grid, refinamiento post-entrenamiento, regularizacion L1/entropia, muestreo adaptativo, descomposicion de dominio -- Secciones 5-9) que no se implementan aqui por simplicidad.
3. **Sin las librerias oficiales**: no se ejecuta el codigo de los repositorios `pykan`, `ChebyKAN` ni `FourierKAN` citados en el documento; cada capa se reimplemento directamente en PyTorch a partir de las formulas exactas (Ecs. 11-14 y 29) para minimizar dependencias y maximizar el control sobre la fidelidad matematica, tal como recomienda la practica habitual en este campo.
4. **Hiperparametros fijos y comparables mas no optimizados**: usamos el mismo numero de "atomos" de base por arista en las tres variantes (grid_size=8 B-splines, grado 8 de Chebyshev, K=8 armonicos de Fourier) y el mismo optimizador/epocas para que la comparacion sea justa, pero no se realizo una busqueda de hiperparametros por variante -- el documento mismo advierte (Sec. 11) que las "basis-vs-basis races" con una sola configuracion "mask deeper structure" y no deben tomarse como veredictos definitivos, solo como ilustracion cualitativa del sesgo inductivo de cada base.